# Q3 - Error analysis by class, source institution, and image quality

The Final ANN's test-set errors, broken down beyond the aggregate confusion matrix already in
`experiments/log.csv` (`final_ann_final_000`). This is legitimate post-`final_eval.py`
reporting, not a new round of model selection: the official test metrics were already
computed and logged once; this notebook re-runs inference on the same already-evaluated test
images purely to see *which* images were wrong and why, per CLAUDE.md's Q3 requirement.
No model, threshold, or hyperparameter decision is made from anything here.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from marco.data import CLASS_ORDER, MarcoDataset, load_split
from marco.model import build_model

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

params = json.load(open(PROJECT_ROOT / "experiments" / "models" / "ann_final_history.json"))["params"]
model = build_model(params["backbone"], fine_tune_degree="layer4")
model.load_state_dict(torch.load(PROJECT_ROOT / "experiments" / "models" / "ann_final.pt", map_location="cpu"))
model.eval()

test_df = load_split(DATA_DIR / "splits.csv", "test").reset_index(drop=True)
loader = DataLoader(MarcoDataset(test_df, DATA_DIR, train=False), batch_size=32, shuffle=False, num_workers=0)

all_preds = []
with torch.no_grad():
    for imgs, labels in loader:
        all_preds.append(torch.softmax(model(imgs), dim=1).argmax(dim=1).numpy())
all_preds = np.concatenate(all_preds)
test_df["pred"] = [CLASS_ORDER[i] for i in all_preds]
test_df["correct"] = test_df["pred"] == test_df["label"]
print(f"overall test accuracy: {test_df['correct'].mean():.3f}")

overall test accuracy: 0.809


## By source institution

In [2]:
by_source = test_df.groupby("source").agg(n=("correct", "size"), accuracy=("correct", "mean"))
by_source

,n,accuracy
source,,
0,51,0.686275
1,343,0.746356
2,334,0.847305
3,1305,0.822989
4,20,0.600000


**Reading this:** the two smallest, most underrepresented training sources (0 and 4 - see
`docs/report_q1_eda.md`'s class/source distributions) have the lowest test accuracy, despite
source 0 having a high Crystals share in its test images. This confirms the EDA's Finding 2
concretely: the model hasn't seen enough examples from these imaging setups to generalise as
well, independent of class mix - directly relevant to the web app's "unseen imagers"
limitation (Q4).

## By image quality (file size quartile, as a resolution/detail proxy)

In [3]:
test_df["filesize_kb"] = [(DATA_DIR / p).stat().st_size / 1024 for p in test_df["path"]]
test_df["size_quartile"] = pd.qcut(
    test_df["filesize_kb"], 4, labels=["Q1 smallest", "Q2", "Q3", "Q4 largest"]
)
by_quality = test_df.groupby("size_quartile", observed=True).agg(n=("correct", "size"), accuracy=("correct", "mean"))
by_quality

,n,accuracy
size_quartile,,
Q1 smallest,514,0.844358
Q2,513,0.824561
Q3,513,0.826511
Q4 largest,513,0.738791


**Reading this:** counter-intuitively, the *largest* image files have the *lowest* accuracy,
not the highest. A plausible explanation is that file size correlates more strongly with
source institution/imaging equipment than with "quality" in the sense of being easier for the
model - large files may simply come from an institution whose imaging style is harder for this
model (consistent with the by-source table above), rather than large files being intrinsically
harder to classify. Reported as an observed, somewhat surprising finding, not over-interpreted
causally.

## By class (full confusion matrix, for reference)

In [4]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_df["label"], test_df["pred"], labels=CLASS_ORDER)
pd.DataFrame(cm, index=CLASS_ORDER, columns=CLASS_ORDER)

,Clear,Crystals,Other,Precipitate
Clear,602,52,17,30
Crystals,5,219,8,22
Other,4,9,91,9
Precipitate,50,93,94,748
